# Tarea 4: Análisis de Datos y Optimización

## **Yajhaira Quiroz**

**Fecha de entrega:** [VER CANVAS]

**Puntaje total:** 20 puntos

**Instrucciones:**
- Completa los tres problemas en este notebook
- Escribe tu código en las celdas indicadas
- Ejecuta todas las celdas para verificar que tu código funciona
- Guarda tu archivo `.ipynb` en la carpeta `tareas` de tu repositorio privado de GitHub (compartido con el docente)
- Envía el enlace a tu notebook en Canvas

**⚠️ IMPORTANTE:** GitHub registra el historial de cambios de cada archivo. Tu notebook debe ser subido a GitHub **antes del plazo**. **NO** modifiques el archivo después del plazo — los cambios tardíos serán detectados y pueden resultar en penalidad.

**Integridad académica:** Esta es una tarea individual. Puedes consultar los materiales del curso, documentación de Python, herramientas de IA y discutir conceptos con compañeros, pero todo el código debe ser tuyo.

---

In [1]:
# Importaciones estándar - ejecuta esta celda primero
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize, curve_fit
from io import StringIO

---
## Problema 1: Análisis de Calidad de Agua en Ríos Andino-Amazónicos (7 puntos)

Estás analizando datos de calidad de agua de estaciones de monitoreo en tres ríos de la cuenca amazónica peruana, provenientes de la red de monitoreo de la **Autoridad Nacional del Agua (ANA)**. El conjunto de datos contiene mediciones de temperatura, oxígeno disuelto (OD), pH y conductividad eléctrica recolectadas durante varios meses.

### Tus Tareas:

**Parte A (2 puntos):** Carga y explora los datos
1. Carga los datos del string CSV provisto abajo en un DataFrame de pandas
2. Muestra información básica del conjunto de datos (forma, tipos de datos, primeras filas)
3. Verifica valores faltantes e indica cuántos hay en cada columna
4. Convierte la columna `fecha` a formato datetime usando `pd.to_datetime()`

**Parte B (3 puntos):** Análisis de datos con agrupación
1. Calcula la media, desviación estándar, mínimo y máximo de oxígeno disuelto (`od_mg_l`) agrupado por `estacion_id`
2. Determina qué estación tiene la media más baja de oxígeno disuelto
3. Crea una nueva columna llamada `estado_od` que clasifique cada medición como:
   - "Crítico" si OD < 4 mg/L
   - "Bajo" si OD está entre 4 y 6 mg/L
   - "Adecuado" si OD está entre 6 y 8 mg/L
   - "Bueno" si OD >= 8 mg/L
4. Cuenta cuántas mediciones caen en cada categoría de `estado_od` por estación

**Parte C (2 puntos):** Filtrado y resumen
1. Filtra los datos para incluir solo mediciones donde temperatura > 20°C Y pH entre 6.5 y 8.5
2. Para este subconjunto filtrado, calcula la conductividad media por mes (pista: extrae el mes de la fecha)
3. Identifica qué combinación estación-mes tuvo el mayor número de lecturas con OD "Crítico" o "Bajo"

In [12]:
# Dataset de calidad de agua - ríos andino-amazónicos del Perú
calidad_agua_csv = (
    "estacion_id,fecha,temp_c,od_mg_l,ph,conductividad_us\n"
    "RIO_UCAYALI,2024-05-15,24.3,7.2,7.1,145\n"
    "RIO_UCAYALI,2024-05-22,25.1,6.8,7.0,152\n"
    "RIO_UCAYALI,2024-06-05,24.8,6.5,6.9,158\n"
    "RIO_UCAYALI,2024-06-19,25.5,5.8,6.8,165\n"
    "RIO_UCAYALI,2024-07-03,24.2,5.2,7.0,172\n"
    "RIO_UCAYALI,2024-07-17,23.8,4.8,7.1,168\n"
    "RIO_UCAYALI,2024-08-01,24.5,4.2,7.2,175\n"
    "RIO_UCAYALI,2024-08-15,25.2,5.0,7.0,169\n"
    "RIO_TAMBOPATA,2024-05-15,21.8,8.5,7.4,98\n"
    "RIO_TAMBOPATA,2024-05-22,22.5,8.1,7.5,105\n"
    "RIO_TAMBOPATA,2024-06-05,22.9,7.8,7.3,112\n"
    "RIO_TAMBOPATA,2024-06-19,23.4,7.2,7.2,118\n"
    "RIO_TAMBOPATA,2024-07-03,22.1,6.8,7.1,125\n"
    "RIO_TAMBOPATA,2024-07-17,21.8,6.5,7.0,121\n"
    "RIO_TAMBOPATA,2024-08-01,22.5,6.9,7.1,128\n"
    "RIO_TAMBOPATA,2024-08-15,23.1,7.2,7.2,115\n"
    "RIO_MANTARO,2024-05-15,13.1,9.5,6.5,312\n"
    "RIO_MANTARO,2024-05-22,14.2,8.8,6.4,325\n"
    "RIO_MANTARO,2024-06-05,12.8,8.2,6.3,338\n"
    "RIO_MANTARO,2024-06-19,13.5,7.5,6.2,352\n"
    "RIO_MANTARO,2024-07-03,11.9,6.8,6.0,368\n"
    "RIO_MANTARO,2024-07-17,10.8,5.9,5.9,378\n"
    "RIO_MANTARO,2024-08-01,11.5,5.2,6.1,385\n"
    "RIO_MANTARO,2024-08-15,12.3,4.8,6.2,372\n"
)

# Parte A: Carga y explora los datos
from io import StringIO
# Pista: Usa pd.read_csv(StringIO(calidad_agua_csv))
data = pd.read_csv(StringIO(calidad_agua_csv))
# Primeras filas
print("Primeras 5 filas")
print("_____________________________")
print(data.info())
print(data.head())
print(" ")

# Información de la tada
print("Información de la data")
print("_____________________________")
print(data.info())
print(" ")

# Información estadistico general
print("Información Estadístico")
print("_____________________________")
print(data.describe())
print(" ")

# Verificar valores faltantes
print("Valores faltantes")
print("_____________________________")
print(data.isnull().sum())
print(" ")

# Convertir fecha
data['fecha'] = pd.to_datetime(data['fecha'])
print(data.head())
print(" ")

Primeras 5 filas
_____________________________
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   estacion_id       24 non-null     object 
 1   fecha             24 non-null     object 
 2   temp_c            24 non-null     float64
 3   od_mg_l           24 non-null     float64
 4   ph                24 non-null     float64
 5   conductividad_us  24 non-null     int64  
dtypes: float64(3), int64(1), object(2)
memory usage: 1.3+ KB
None
   estacion_id       fecha  temp_c  od_mg_l   ph  conductividad_us
0  RIO_UCAYALI  2024-05-15    24.3      7.2  7.1               145
1  RIO_UCAYALI  2024-05-22    25.1      6.8  7.0               152
2  RIO_UCAYALI  2024-06-05    24.8      6.5  6.9               158
3  RIO_UCAYALI  2024-06-19    25.5      5.8  6.8               165
4  RIO_UCAYALI  2024-07-03    24.2      5.2  7.0               172
 
Info

In [23]:
# Parte B: Análisis de datos con agrupación
# 1. Calcula la media, desviación estándar, mínimo y máximo de oxígeno disuelto (od_mg_l) agrupado por estacion_id
media = data.groupby('estacion_id')['od_mg_l'].mean()
desviacion_estandar = data.groupby('estacion_id')['od_mg_l'].std()
minimo = data.groupby('estacion_id')['od_mg_l'].min()
maximo = data.groupby('estacion_id')['od_mg_l'].max()

# 2. Determina qué estación tiene la media más baja de oxígeno disuelto
estacion_minima_media = media.idxmin()
print(f"La estación con la media más baja de oxígeno disuelto es: {estacion_minima_media}")
print(" ")
# 3. Crea una nueva columna llamada estado_od que clasifique cada medición
data['estado_od'] = pd.cut(data['od_mg_l'], bins=[-np.inf, 4, 6, 8, np.inf], labels=['Crítico', 'Bajo', 'Adecuado', 'Bueno'])
print(data.head())

# Cuenta cuántas mediciones caen en cada categoría de estado_od por estación
conteo_por_estado = data.groupby(['estacion_id', 'estado_od']).size().unstack(fill_value=0)
print(conteo_por_estado)
print(" ")

La estación con la media más baja de oxígeno disuelto es: RIO_UCAYALI
 
   estacion_id      fecha  temp_c  od_mg_l   ph  conductividad_us estado_od
0  RIO_UCAYALI 2024-05-15    24.3      7.2  7.1               145  Adecuado
1  RIO_UCAYALI 2024-05-22    25.1      6.8  7.0               152  Adecuado
2  RIO_UCAYALI 2024-06-05    24.8      6.5  6.9               158  Adecuado
3  RIO_UCAYALI 2024-06-19    25.5      5.8  6.8               165      Bajo
4  RIO_UCAYALI 2024-07-03    24.2      5.2  7.0               172      Bajo
estado_od      Crítico  Bajo  Adecuado  Bueno
estacion_id                                  
RIO_MANTARO          0     3         2      3
RIO_TAMBOPATA        0     0         6      2
RIO_UCAYALI          0     5         3      0
 


/tmp/ipykernel_8896/2928913865.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  conteo_por_estado = data.groupby(['estacion_id', 'estado_od']).size().unstack(fill_value=0)


In [28]:
# Parte C: Filtrado y resumen
# 1. Filtra los datos para incluir solo mediciones donde temperatura > 20°C Y pH entre 6.5 y 8.5
datos_filtrados = data[(data['temp_c'] > 20) & (data['ph'] >= 6.5) & (data['ph'] <= 8.5)]
# 2. Para este subconjunto filtrado, calcula la conductividad media por mes (pista: extrae el mes de la fecha)
datos_filtrados['mes'] = datos_filtrados['fecha'].dt.month
conductividad_media_por_mes = datos_filtrados.groupby('mes')['conductividad_us'].mean()
print(conductividad_media_por_mes)
# 3. Identifica qué combinación estación-mes tuvo el mayor número de lecturas con OD "Crítico" o "Bajo"
data['mes'] = data['fecha'].dt.month
estacion_mes_max_critico_bajo = data[(data['estado_od'].isin(['Crítico', 'Bajo']))].groupby(['estacion_id', 'mes']).size().idxmax()
print(f"La combinación estación-mes con el mayor número de lecturas con OD 'Crítico' o 'Bajo' es: {estacion_mes_max_critico_bajo}")
print(" ")

mes
5    125.00
6    138.25
7    146.50
8    146.75
Name: conductividad_us, dtype: float64
La combinación estación-mes con el mayor número de lecturas con OD 'Crítico' o 'Bajo' es: ('RIO_MANTARO', np.int32(8))
 


/tmp/ipykernel_8896/1191465970.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datos_filtrados['mes'] = datos_filtrados['fecha'].dt.month


---
## Problema 2: Comparación Estadística de Parcelas Forestales en la Amazonía (6 puntos)

Investigadores del **INIA (Instituto Nacional de Innovación Agraria) - Estación Experimental Pucallpa** midieron la biomasa arbórea (kg) en parcelas pareadas — unas sometidas a un tratamiento de aprovechamiento forestal de impacto reducido (AFIR) y otras dejadas como control. Se quiere determinar si el tratamiento afectó significativamente la biomasa individual de los árboles y si existe relación entre el diámetro y la biomasa.

### Tus Tareas:

**Parte A (2 puntos):** Comparación de grupos de tratamiento
1. Calcula estadísticas descriptivas (media, desviación estándar, mediana) de biomasa para cada grupo
2. Realiza una prueba t de dos muestras independientes para determinar si hay diferencia significativa en la biomasa media entre parcelas control y AFIR (α = 0.05)
3. Plantea tu hipótesis nula y alternativa, reporta el estadístico t y el p-valor, y escribe una conclusión

**Parte B (2 puntos):** Análisis de correlación
1. Calcula el coeficiente de correlación de Pearson entre el DAP y la biomasa para todo el conjunto de datos
2. Evalúa si esta correlación es estadísticamente significativa (α = 0.05)
3. Interpreta la fuerza y dirección de la correlación

**Parte C (2 puntos):** Ajuste de distribución
1. Ajusta una distribución normal a los datos de biomasa de las parcelas control
2. Reporta los parámetros ajustados (μ y σ)
3. Calcula la probabilidad de que un árbol seleccionado aleatoriamente de las parcelas control tenga biomasa > 150 kg
4. ¿Qué valor de biomasa representa el percentil 90 para los árboles de parcelas control?

In [30]:
# Datos de parcelas forestales - INIA Pucallpa
np.random.seed(458)  # Para reproducibilidad

# Parcelas control: bosque sin intervención
n_control = 35
dap_control = np.random.uniform(15, 50, n_control)  # DAP en cm
biomasa_control = 0.1 * dap_control**2.2 + np.random.normal(0, 15, n_control)
biomasa_control = np.maximum(biomasa_control, 10)  # Asegurar valores positivos

# Parcelas AFIR: los árboles remanentes disponen de más recursos
n_afir = 30
dap_afir = np.random.uniform(18, 55, n_afir)  # DAP en cm
biomasa_afir = 0.12 * dap_afir**2.2 + np.random.normal(5, 18, n_afir)
biomasa_afir = np.maximum(biomasa_afir, 10)

# Crear DataFrame
bosque_df = pd.DataFrame({
    'dap_cm': np.concatenate([dap_control, dap_afir]),
    'biomasa_kg': np.concatenate([biomasa_control, biomasa_afir]),
    'tratamiento': ['Control']*n_control + ['AFIR']*n_afir
})

print(bosque_df.head())
print(f"\nEspecies representativas: Caoba (Swietenia macrophylla), Cedro (Cedrela odorata), Tornillo (Cedrelinga cateniformis)")

      dap_cm  biomasa_kg tratamiento
0  43.208835  395.956074     Control
1  49.475851  521.291644     Control
2  19.647405   54.799539     Control
3  23.651882  119.264060     Control
4  40.431164  335.576313     Control

Especies representativas: Caoba (Swietenia macrophylla), Cedro (Cedrela odorata), Tornillo (Cedrelinga cateniformis)


In [36]:
# Parte A: Comparación de grupos de tratamiento
# 1. Calcula estadísticas descriptivas (media, desviación estándar, mediana) de biomasa para cada grupo
media_control = bosque_df[bosque_df['tratamiento'] == 'Control']['biomasa_kg'].mean()
media_afir = bosque_df[bosque_df['tratamiento'] == 'AFIR']['biomasa_kg'].mean()
desviacion_estandar_control = bosque_df[bosque_df['tratamiento'] == 'Control']['biomasa_kg'].std
desviacion_estandar_afir = bosque_df[bosque_df['tratamiento'] == 'AFIR']['biomasa_kg'].std
mediana_control = bosque_df[bosque_df['tratamiento'] == 'Control']['biomasa_kg'].median()
mediana_afir = bosque_df[bosque_df['tratamiento'] == 'AFIR']['biomasa_kg'].median()

# 2. Realiza una prueba t de dos muestras independientes para determinar si hay diferencia significativa en la biomasa media entre parcelas control y AFIR (α = 0.05)
biomasa_control = bosque_df[bosque_df['tratamiento'] == 'Control']['biomasa_kg']
biomasa_afir = bosque_df[bosque_df['tratamiento'] == 'AFIR']['biomasa_kg']

t_stat, p_valor = stats.ttest_ind(biomasa_control, biomasa_afir, equal_var=True)
print(f"Estadístico t: {t_stat:.4f}")
print(f"Valor p (p-value): {p_valor:.4f}")

alpha = 0.05
print(" ")

# 3. Plantea tu hipótesis nula y alternativa, reporta el estadístico t y el p-valor, y escribe una conclusión
if p_valor < alpha:
    print(f" Como el valor p ({p_valor:.4f}) < alpha (0.05), SE RECHAZA la hipótesis nula (H0).")
    print(" Existe suficiente evidencia estadística para afirmar que hay una diferencia")
    print(" significativa en la biomasa media entre las parcelas Control y AFIR de INIA Pucallpa.")
else:
    print(f" Como el valor p ({p_valor:.4f}) >= alpha (0.05), NO SE RECHAZA la hipótesis nula (H0).")
    print(" No existe suficiente evidencia estadística para afirmar que hay una diferencia")
    print(" significativa en la biomasa media entre las parcelas Control y AFIR.")


Estadístico t: -1.2070
Valor p (p-value): 0.2319
 
 Como el valor p (0.2319) >= alpha (0.05), NO SE RECHAZA la hipótesis nula (H0).
 No existe suficiente evidencia estadística para afirmar que hay una diferencia
 significativa en la biomasa media entre las parcelas Control y AFIR.


In [40]:
# Parte B: Análisis de correlación
# 1. Calcula el coeficiente de correlación de Pearson entre el DAP y la biomasa para todo el conjunto de datos
correlacion_pearson = bosque_df['dap_cm'].corr(bosque_df['biomasa_kg'])
print(f"Coeficiente de correlación de Pearson: {correlacion_pearson:.4f}")

# 2. Evalúa si esta correlación es estadísticamente significativa (α = 0.05)
r_val, p_valor_corr = stats.pearsonr(bosque_df['dap_cm'], bosque_df['biomasa_kg'])

alpha = 0.05
if p_valor_corr < alpha:
    print(f" Resultado: Como el valor p < alpha, la correlación SÍ es estadísticamente significativa.")
else:
    print(f" Resultado: Como el valor p >= alpha, la correlación NO es estadísticamente significativa.")

# 3. Interpreta la fuerza y dirección de la correlación
direccion = "positiva" if correlacion_pearson > 0 else "negativa"

r_abs = abs(correlacion_pearson)
if r_abs >= 0.9:
    fuerza = "muy fuerte"
elif r_abs >= 0.7:
    fuerza = "fuerte"
elif r_abs >= 0.4:
    fuerza = "moderada"
else:
    fuerza = "débil"

print(f" - Dirección: Es una correlación {direccion}.")
print(f" - Fuerza: Es una relación {fuerza} (r = {correlacion_pearson:.4f}).")

Coeficiente de correlación de Pearson: 0.9565
 Resultado: Como el valor p < alpha, la correlación SÍ es estadísticamente significativa.
 - Dirección: Es una correlación positiva.
 - Fuerza: Es una relación muy fuerte (r = 0.9565).


In [46]:
# Parte C: Ajuste de distribución
# 1. Ajusta una distribución normal a los datos de biomasa de las parcelas control
mu_ajustado, sigma_ajustado = stats.norm.fit(biomasa_control)

print("PARÁMETROS AJUSTADOS DE LA DISTRIBUCIÓN NORMAL:")
print(f"Media estimada (μ):           {mu_ajustado:.4f} kg")
print(f"Desviación estándar (σ):     {sigma_ajustado:.4f} kg\n")

# 2. Calcular la probabilidad de que un árbol tenga biomasa > 150 kg
probabilidad_mayor_150 = stats.norm.sf(150, loc=mu_ajustado, scale=sigma_ajustado)

print("CÁLCULO DE PROBABILIDAD (Biomasa > 150 kg):")
print(f"P(X > 150 kg):                {probabilidad_mayor_150:.4f}")
print(f"En porcentaje:                {probabilidad_mayor_150 * 100:.2f}%\n")

# 3. ¿Qué valor de biomasa representa el percentil 90?
percentil_90 = stats.norm.ppf(0.90, loc=mu_ajustado, scale=sigma_ajustado)

print("PERCENTIL 90 DE BIOMASA:")
print(f"Valor del percentil 90:       {percentil_90:.4f} kg")
print(f"Interpretación: El 90% de los árboles control tiene una biomasa")
print(f"igual o menor a este valor, y solo el 10% lo supera.")

PARÁMETROS AJUSTADOS DE LA DISTRIBUCIÓN NORMAL:
Media estimada (μ):           269.6426 kg
Desviación estándar (σ):     155.9335 kg

CÁLCULO DE PROBABILIDAD (Biomasa > 150 kg):
P(X > 150 kg):                0.7785
En porcentaje:                77.85%

PERCENTIL 90 DE BIOMASA:
Valor del percentil 90:       469.4795 kg
Interpretación: El 90% de los árboles control tiene una biomasa
igual o menor a este valor, y solo el 10% lo supera.


---
## Problema 3: Ajuste de Curva de Respuesta a la Luz (7 puntos)

La fotosíntesis depende de la intensidad de luz siguiendo una curva de saturación. La **hipérbola rectangular** se usa comúnmente para modelar esta relación:

$$A = \frac{A_{max} \cdot I}{K + I} - R_d$$

Donde:
- $A$ = tasa de fotosíntesis neta (μmol CO₂ m⁻² s⁻¹)
- $A_{max}$ = tasa máxima de fotosíntesis a saturación de luz
- $I$ = intensidad de luz (μmol fotones m⁻² s⁻¹, PAR)
- $K$ = constante de media saturación (nivel de luz en el que A = A_max/2 - R_d)
- $R_d$ = tasa de respiración en oscuridad (CO₂ liberado cuando I = 0)

Los datos provienen de mediciones de *Cecropia sp.* ("cetico"), una especie pionera característica de la Amazonía peruana, muy importante en la regeneración de bosques perturbados.

### Tus Tareas:

**Parte A (2 puntos):** Define el modelo y la función de costo
1. Escribe una función `respuesta_luz(I, Amax, K, Rd)` que implemente la ecuación anterior
2. Escribe una función de costo `respuesta_luz_mse(params, I_datos, A_datos)` que calcule el error cuadrático medio entre las tasas de fotosíntesis observadas y predichas
3. Prueba tu función `respuesta_luz` calculando A para I = 500 con Amax=25, K=200, Rd=2

**Parte B (3 puntos):** Ajusta el modelo usando optimización
1. Usa `scipy.optimize.minimize` para encontrar los parámetros óptimos (Amax, K, Rd) que minimicen el MSE
2. Usa valores iniciales: Amax=20, K=150, Rd=1
3. Reporta los parámetros ajustados y el MSE final
4. Ajusta también el modelo usando `scipy.optimize.curve_fit` y compara los resultados

**Parte C (2 puntos):** Evalúa e interpreta el modelo
1. Calcula los valores de fotosíntesis predichos usando tus parámetros ajustados
2. Calcula R² (coeficiente de determinación) para evaluar el ajuste del modelo:
   $$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$
3. Calcula el **punto de compensación lumínico** (el nivel de luz donde A = 0, es decir, la fotosíntesis iguala a la respiración). Pista: despeja I cuando A = 0
4. ¿Cuál es la tasa de fotosíntesis a saturación lumínica (Amax - Rd)?

In [47]:
# Datos de curva de respuesta a la luz - Cecropia sp. (cetico)
# Mediciones en parcela de investigación, Madre de Dios

# PAR (radiación fotosintéticamente activa) en μmol fotones m⁻² s⁻¹
par_datos = np.array([0, 25, 50, 75, 100, 150, 200, 300, 400, 600, 800, 1000, 1200, 1500, 1800])

# Tasa de fotosíntesis neta en μmol CO₂ m⁻² s⁻¹
foto_datos = np.array([-1.8, 1.2, 4.5, 7.1, 9.2, 12.5, 14.8, 17.5, 19.2, 21.1, 22.0, 22.5, 22.8, 23.0, 23.1])

print(f"Rango PAR: {par_datos.min()} a {par_datos.max()} μmol fotones m⁻² s⁻¹")
print(f"Rango fotosíntesis: {foto_datos.min()} a {foto_datos.max()} μmol CO₂ m⁻² s⁻¹")
print("Especie: Cecropia sp. (cetico) - pionera amazónica")

Rango PAR: 0 a 1800 μmol fotones m⁻² s⁻¹
Rango fotosíntesis: -1.8 a 23.1 μmol CO₂ m⁻² s⁻¹
Especie: Cecropia sp. (cetico) - pionera amazónica


In [48]:
# Parte A: Define el modelo y la función de costo
# 1. Definición del modelo de hipérbola rectangular
def respuesta_luz(I, Amax, K, Rd):
    return (Amax * I) / (K + I) - Rd

# 2. Función de costo: Error Cuadrático Medio (MSE)
def respuesta_luz_mse(params, I_datos, A_datos):
    Amax, K, Rd = params
    predichos = respuesta_luz(I_datos, Amax, K, Rd)
    mse = np.mean((A_datos - predichos) ** 2)
    return mse

# 3. Prueba de la función para I = 500
A_prueba = respuesta_luz(500, Amax=25, K=200, Rd=2)
print(f"3. Prueba del modelo para I = 500: A = {A_prueba:.4f} μmol CO₂ m⁻² s⁻¹\n")


3. Prueba del modelo para I = 500: A = 15.8571 μmol CO₂ m⁻² s⁻¹



In [49]:
# Parte B: Ajusta el modelo usando optimización
params_iniciales = [20, 150, 1]

# 1. Optimización con scipy.optimize.minimize
resultado_minimize = minimize(
    respuesta_luz_mse,
    x0=params_iniciales,
    args=(par_datos, foto_datos),
    method='Nelder-Mead'
)
Amax_min, K_min, Rd_min = resultado_minimize.x
mse_final = resultado_minimize.fun

# 3. Reportar parámetros de minimize
print("1 y 3. Resultados usando 'minimize':")
print(f"   - Amax:    {Amax_min:.4f}")
print(f"   - K:       {K_min:.4f}")
print(f"   - Rd:      {Rd_min:.4f}")
print(f"   - MSE:     {mse_final:.4f}\n")

# 4. Optimización alternativa con scipy.optimize.curve_fit
params_opt_cf, _ = curve_fit(respuesta_luz, par_datos, foto_datos, p0=params_iniciales)
Amax_cf, K_cf, Rd_cf = params_opt_cf
mse_cf = np.mean((foto_datos - respuesta_luz(par_datos, *params_opt_cf)) ** 2)

print("4. Resultados usando 'curve_fit' (Comparación):")
print(f"   - Amax:    {Amax_cf:.4f}")
print(f"   - K:       {K_cf:.4f}")
print(f"   - Rd:      {Rd_cf:.4f}")
print(f"   - MSE:     {mse_cf:.4f}")


1 y 3. Resultados usando 'minimize':
   - Amax:    28.4460
   - K:       134.7544
   - Rd:      2.6272
   - MSE:     0.2373

4. Resultados usando 'curve_fit' (Comparación):
   - Amax:    28.4460
   - K:       134.7545
   - Rd:      2.6272
   - MSE:     0.2373


In [54]:
# Parte C: Evalúa e interpreta el modelo

Amax_opt, K_opt, Rd_opt = Amax_cf, K_cf, Rd_cf

# 1. Calcular valores de fotosíntesis predichos
foto_predichos = respuesta_luz(par_datos, Amax_opt, K_opt, Rd_opt)

# 2. Calcular el coeficiente de determinación R²
ss_res = np.sum((foto_datos - foto_predichos) ** 2)
ss_tot = np.sum((foto_datos - np.mean(foto_datos)) ** 2)
r_cuadrado = 1 - (ss_res / ss_tot)
print(f"2. Coeficiente de determinación R²: {r_cuadrado:.4f}")

# 3. Calcular el punto de compensación lumínico (A = 0)
# Despejando: I = (Rd * K) / (Amax - Rd)
punto_compensacion = (Rd_opt * K_opt) / (Amax_opt - Rd_opt)
print(f"3. Punto de compensación lumínico: {punto_compensacion:.4f} μmol fotones")

# 4. Calcular la tasa de fotosíntesis a saturación lumínica (Amax - Rd)
fotosintesis_saturacion = Amax_opt - Rd_opt
print(f"4. Fotosíntesis neta máxima a saturación (Amax - Rd): {fotosintesis_saturacion:.4f}")


2. Coeficiente de determinación R²: 0.9966
3. Punto de compensación lumínico: 13.7117 μmol fotones
4. Fotosíntesis neta máxima a saturación (Amax - Rd): 25.8189


---
## Lista de verificación para entrega

Antes de entregar, verifica que:

- [ ] Todas las celdas de código se ejecutan sin errores
- [ ] Los tres problemas están completos
- [ ] Los resultados son visibles en todas las celdas
- [ ] Tu nombre está incluido al inicio
- [ ] El archivo está guardado en la carpeta `tareas` de tu repositorio privado de GitHub
- [ ] El archivo está subido a GitHub **antes del plazo**
- [ ] El enlace a tu notebook está enviado en Canvas